Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Allow importing directly from the src/ directory
sys.path.append("..")

from src.utils.config_loader import load_config
from src.utils.logger import get_logger
from src.extraction.pdf_extractor import extract_text_from_pdf
from src.extraction.docx_extractor import extract_text_from_docx
from src.extraction.txt_extractor import extract_text_from_txt
from src.extraction.ocr_extractor import extract_text_from_image, extract_text_from_scanned_pdf
from src.extraction.extractor import extract_text

# Initialize config and logger
config = load_config()
logger = get_logger("Notebook-01-Extraction")

print("=== CONFIGURATION LOADED ===")
print(f"Supported formats : {config['extraction']['supported_formats']}")
print(f"OCR DPI           : {config['extraction']['ocr_dpi']}")
print(f"Native PDF cutoff : {config['extraction']['min_text_length_for_native_pdf']} chars")

Unit Testing Individual Format Extractors

In [ ]:
# Helper function to print standardized extractor outputs
def preview_extraction(title: str, result: dict):
    print(f"\n{'='*50}")
    print(f" {title}")
    print(f"{'='*50}")
    print(f"Metadata   : {result.get('metadata', {})}")
    print(f"Character Count : {result.get('metadata', {}).get('char_count', 0)}")
    preview = result.get("text", "").strip().replace("\n", " ")
    print(f"Text Preview    : {preview[:200]}..." if preview else "Text Preview    : [EMPTY]")

samples_dir = Path("../data/samples")

# 1. Native PDF Test
pdf_res = extract_text_from_pdf(str(samples_dir / "ANS_PDF.pdf"))
preview_extraction("PDF EXTRACTOR (Native)", pdf_res)

# 2. DOCX Test
docx_res = extract_text_from_docx(str(samples_dir / "ANS_DOCX.docx"))
preview_extraction("DOCX EXTRACTOR", docx_res)

# 3. TXT Test
txt_res = extract_text_from_txt(str(samples_dir / "ANS_TXT.txt"))
preview_extraction("TXT EXTRACTOR", txt_res)

# 4. Image OCR Test
png_res = extract_text_from_image(str(samples_dir / "ANS_PNG.png"))
preview_extraction("IMAGE OCR EXTRACTOR", png_res)

Full Dispatcher Batch Test (extract_text)

In [ ]:
import os

print("\n=== RUNNING UNIFIED DISPATCHER ACROSS ALL SAMPLE FILES ===")
sample_files = sorted([f for f in os.listdir(samples_dir) if not f.startswith(".")])

for fname in sample_files:
    file_path = samples_dir / fname
    result = extract_text(str(file_path))
    
    status = "ERROR" if "error" in result["metadata"] else "✅ OK"
    char_count = result["metadata"].get("char_count", 0)
    engine = result["metadata"].get("engine", "native")
    
    print(f"{fname:<20} | Status: {status:<8} | Chars: {char_count:<5} | Engine: {engine}")

Edge Case & Graceful Fallback Verification

In [ ]:
# Test 1: Non-existent file (should return structured error, not crash)
missing_res = extract_text("data/samples/does_not_exist.pdf")
print("1. Missing File Error Dict:")
print(missing_res)

# Test 2: Unsupported extension
dummy_invalid = Path("../data/samples/invalid_format.xyz")
dummy_invalid.write_text("invalid content", encoding="utf-8")

invalid_res = extract_text(str(dummy_invalid))
print("\n2. Unsupported Format Error Dict:")
print(invalid_res)

# Cleanup dummy file
if dummy_invalid.exists():
    dummy_invalid.unlink()